# Neural Machine Translation

In [299]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Setting this env variable prevents TF warnings from showing up
import tensorflow as tf
from utils import (sentences, train_data, val_data, english_vectorizer, portuguese_vectorizer, 
                   masked_loss, masked_acc, tokens_to_text)
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset

In [300]:
# Data gốc
portuguese_sentences, english_sentences = sentences # len=190639
# portuguese_sentences[-5]='Não importa o quanto você tenta convencer os outros de que chocolate é baunilha, ele ainda será chocolate, mesmo que você possa convencer a si mesmo e poucos outros de que é baunilha.'
# english_sentences[-5]="No matter how much you try to convince people that chocolate is vanilla, it'll still be chocolate, even though you may manage to convince yourself and a few others that it's vanilla."

In [301]:
# Đã có english_vectorizer và portuguese_vectorizer
print(english_vectorizer.get_vocabulary())  # len=12000
print(portuguese_vectorizer.get_vocabulary()) # len=12000

['', '[UNK]', '[SOS]', '[EOS]', '.', 'tom', 'i', 'to', 'you', 'the', '?', 'a', 'is', 'that', 'do', 'in', 'was', 'me', 'dont', 'have', 'this', 'he', 'of', 'it', ',', 'my', 'im', 'what', 'know', 'we', 'are', 'mary', 'for', 'be', 'want', 'like', 'your', 'with', 'on', 'think', 'didnt', 'not', 'did', 'and', 'his', 'can', 'go', 'at', 'were', 'how', 'here', 'going', 'its', 'has', 'she', 'very', 'why', 'about', 'youre', 'will', 'all', 'time', 'they', 'there', 'one', 'as', 'need', 'get', 'cant', 'isnt', 'who', 'doesnt', 'boston', 'him', 'french', 'ill', 'if', 'had', 'said', 'when', 'up', 'help', 'would', 'out', 'where', 'toms', 'tell', 'ive', 'no', 'us', 'good', 'been', 'an', 'never', 'from', 'see', 'by', 'just', 'come', 'than', 'her', 'told', 'now', 'still', 'really', 'so', 'doing', 'got', 'much', 'please', 'should', 'well', 'more', 'thats', 'home', 'something', 'wont', 'too', 'lot', 'wanted', 'some', 'car', 'could', 'back', 'work', 'anything', 'wasnt', 'three', 'last', 'does', 'take', 'people

In [302]:
# This helps you convert from words to ids
word_to_id = tf.keras.layers.StringLookup(
    vocabulary=portuguese_vectorizer.get_vocabulary(), 
    mask_token="", 
    oov_token="[UNK]"
)

# This helps you convert from ids to words
id_to_word = tf.keras.layers.StringLookup(
    vocabulary=portuguese_vectorizer.get_vocabulary(),
    mask_token="",
    oov_token="[UNK]",
    invert=True,
)

In [303]:
# Data đã đc preprocessed
# train_data chứa 2385 elements, mỗi element chứa ba Tensors to_translate, sr_translation, translation, có shape (64, n1), (64, n2), (64, n2).
# Ở đây 64 là batch size, n1 là độ dài câu English, n2 là độ dài câu Portugese tương ứng
# to_translate và sr_translation đều bắt đầu bằng SOS có id=2
for (to_translate, sr_translation), translation in train_data.take(1):
    print(f"Tokenized english sentence:\n{to_translate[0, :]}\n")
    print(f"Tokenized portuguese sentence (shifted to the right):\n{sr_translation[0, :]}\n")
    print(f"Tokenized portuguese sentence:\n{translation[0, :]}")

Tokenized english sentence:
[  2  75  33 188   7  46   4   3   0   0   0   0   0   0   0]

Tokenized portuguese sentence (shifted to the right):
[   2 1061  159   22   62    4    0    0    0    0    0    0    0    0]

Tokenized portuguese sentence:
[1061  159   22   62    4    3    0    0    0    0    0    0    0    0]


## 2. Model
Ta sẽ dùng model có dạng encoder-decoder architecture. Ta có thể dùng RNN ở phần tam giác ở giữa, sẽ nhận tokenized version of a sentence từ encoder rồi đưa sang decoder để dịch. Nếu chỉ dùng normal seq2seq model thì chỉ hoạt động tốt với câu ngắn và vừa. Để khắc phục ta dùng attention, cho phép decoder truy cập tất cả các phần của input sentence. VD dịch câu "how are you to day" sang Portuguese, và đã gen ra word "como". Attention sẽ score từng encoder hidden state (các hcn màu cam) để biết decoder nên tập trung vào state nào để gen ra word tiếp theo. Trong quá trình training, model sẽ biết nó nên tập trung vào state "are" của encoder và gán XS cao cho word "você".

<img src="images/plain_rnn.png" height=250/>
<img src="images/attention_overview.png" height=350/>

Ở đây ta dùng Scaled Dot Product Attention:
$$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$
Nó tính scores dùng queries (Q) và keys (K), rồi nhân với values (V) để có một context vector tại một timestep of decoder. Context vector này sau đó đc đưa vào decoder RNN để tính các probabilities của next predicted word. Ở đây chia cho square root of keys dimensionality ($\sqrt{d_k}$) là để improve model performance. Ta sẽ dùng encoder activations (i.e. encoder hidden states) làm cả keys và values, và dùng decoder activations (i.e. decoder hidden states) làm queries.

<img src="images/NMTModel.png" height=1000/>

In [304]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, units):
        super(Encoder, self).__init__()

        self.embedding = tf.keras.layers.Embedding(  
            input_dim=vocab_size,
            output_dim=units,
            mask_zero=True
        )  

        self.rnn = tf.keras.layers.Bidirectional(  
            merge_mode="sum",  
            layer=tf.keras.layers.LSTM(
                units=units,
                return_sequences=True
            ),  
        )  

    def call(self, context): # Xét context là một vector dài n1
        # Tính Embedding từ context, có embedding dim=256
        # Pass the context through the embedding layer
        x = self.embedding(context) # shape=(n1, 256)

        # Đưa qua Bidirectional LSTM có số units=256, lấy ra tất cả hidden states.
        # Pass the output of the embedding through the RNN
        x = self.rnn(x) # shape=(n1, 256)

        return x
    

class CrossAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()

        self.mha = ( 
            tf.keras.layers.MultiHeadAttention(
                key_dim=units,
                num_heads=1
            ) 
        )  

        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()

    def call(self, context, target):
        # Cho context (đúng là encoded_context, shape=(n1, 256)) và target (đúng là embedding của target, shape=(n2, 256)) vào MultiHeadAttention (ở đây num_heads=1 nên coi như Attention bình thường)
        # Có query là target, key và value đều là encoded_context. Tính theo công thức, ta có QK^T có shape (n2, n1), nên output của MultiHeadAttention có shape (n2, 256) 
        # Call the MH attention by passing in the query and value
        # For this case the query should be the translation and the value the encoded sentence to translate
        # Hint: Check the call arguments of MultiHeadAttention in the docs
        attn_output = self.mha(
            query=target,
            value=context
        ) # shape=(n2, 256)

        # Cộng target với attn_output theo elment-wise
        x = self.add([target, attn_output]) # shape=(n2, 256)

        x = self.layernorm(x) # shape=(n2, 256)

        return x
    

class Decoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, units):
        super(Decoder, self).__init__()

        # The embedding layer
        self.embedding = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=units,
            mask_zero=True
        )  

        # The RNN before attention
        self.pre_attention_rnn = tf.keras.layers.LSTM(
            units=units,
            return_sequences=True,
            return_state=True
        )  

        # The attention layer
        self.attention = CrossAttention(units)

        # The RNN after attention
        self.post_attention_rnn = tf.keras.layers.LSTM(
            units=units,
            return_sequences=True
        )  

        # The dense layer with logsoftmax activation
        self.output_layer = tf.keras.layers.Dense(
            units=vocab_size,
            activation=tf.nn.log_softmax
        )  

    def call(self, context, target, state=None, return_state=False):
        # Tính Embedding từ target, có embedding dim=256
        # Get the embedding of the input
        x = self.embedding(target) # shape=(n2, 256)

        # Cho x vào LSTM có số units=256, lấy ra tất cả hidden states, last hidden state, last cell state.
        # Pass the embedded input into the pre attention LSTM
        # - The LSTM you defined earlier should return the output alongside the state (made up of two tensors)
        # - Pass in the state to the LSTM (needed for inference)
        x, hidden_state, cell_state = self.pre_attention_rnn(x, initial_state=state)
        # x.shape=(n2, 256), hidden_state và cell_state là vector có length=256
        
        # Cho context (đúng là encoded_context) và x vào một khối CrossAttention. Xem tiếp ở CrossAttention
        # Perform cross attention between the context and the output of the LSTM (in that order)
        x = self.attention(context, x) # shape=(n2, 256)

        # Cho x qua một LSTM nữa có số units=256, lấy ra tất cả hidden states.
        # Do a pass through the post attention LSTM
        x = self.post_attention_rnn(x) # shape=(n2, 256)

        # Cho x qua FC log_softmax với số units=vocab_size=12000
        # Compute the logits
        logits = self.output_layer(x) # shape=(n2, 12000)

        if return_state:
            return logits, [hidden_state, cell_state]

        return logits

In [305]:
class Translator(tf.keras.Model):
    def __init__(self, vocab_size, units):
        super().__init__()

        # Define the encoder with the appropriate vocab_size and number of units
        self.encoder = Encoder(vocab_size, units)

        # Define the decoder with the appropriate vocab_size and number of units
        self.decoder = Decoder(vocab_size, units)

    def call(self, inputs):
        # Input của model là cặp to_translate (ở đây là context) và sr_translation (ở đây là target). Xét một sample ta sẽ có hai vectors dài n1 và n2.
        # In this case inputs is a tuple consisting of the context and the target, unpack it into single variables
        context, target = inputs 

        # Đưa context và target vào Encoder. Xem tiếp ở Encoder.
        # Pass the context through the encoder
        encoded_context = self.encoder(context) # shape=(n1, 256)

        # Đưa encoded_context vào decoder. Xem tiếp ở Decoder.
        # Compute the logits by passing the encoded context and the target to the decoder
        logits = self.decoder(encoded_context, target) # shape=(n2, 256)

        return logits

In [306]:
VOCAB_SIZE = 12000
UNITS = 5
model=Translator(VOCAB_SIZE, UNITS)

In [307]:

model.compile(optimizer="adam", loss=masked_loss, metrics=[masked_acc, masked_loss])
model.fit(
    train_data.repeat(),
    epochs=20,
    steps_per_epoch=500,
    validation_data=val_data,
    validation_steps=50,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)],
)

Epoch 1/20


500/500 ━━━━━━━━━━━━━━━━━━━━ 59s 103ms/step - loss: 8.4821 - masked_acc: 0.0856 - masked_loss: 8.4821 - val_loss: 6.2830 - val_masked_acc: 0.1235 - val_masked_loss: 6.2830
Epoch 2/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 52s 104ms/step - loss: 6.0425 - masked_acc: 0.1238 - masked_loss: 6.0425 - val_loss: 5.5639 - val_masked_acc: 0.1248 - val_masked_loss: 5.5639
Epoch 3/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 48s 97ms/step - loss: 5.5674 - masked_acc: 0.1241 - masked_loss: 5.5674 - val_loss: 5.5057 - val_masked_acc: 0.1242 - val_masked_loss: 5.5057
Epoch 4/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 47s 93ms/step - loss: 5.5148 - masked_acc: 0.1240 - masked_loss: 5.5148 - val_loss: 5.3283 - val_masked_acc: 0.1241 - val_masked_loss: 5.3283
Epoch 5/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 50s 100ms/step - loss: 5.3413 - masked_acc: 0.1241 - masked_loss: 5.3413 - val_loss: 5.2255 - val_masked_acc: 0.1252 - val_masked_loss: 5.2255
Epoch 6/20
500/500 ━━━━━━━━━━━━━━━━━━━━ 49s 97ms/step - loss: 5.2278 - masked_acc: 0.1237 - masked

## 3. Translation 

In [308]:
unk_id = word_to_id("[UNK]")
sos_id = word_to_id("[SOS]")
eos_id = word_to_id("[EOS]")

def generate_next_token(decoder, context, next_token, done, state, temperature=0.0):
    # Đưa context shape=(5, 256) và next_token shape=(1, 1) vào decoder
    # Get the logits and state from the decoder
    logits, state = decoder(context, next_token, state=state, return_state=True)
    # logits.shape=(1, 1, 12000)
    
    # Trim the intermediate dimension 
    logits = logits[:, -1, :] # shape=(1, 12000)
    
    # Nếu temp=0 thì next_token là vị trí có logits lớn nhất
    # If temp is 0 then next_token is the argmax of logits
    if temperature == 0.0:
        next_token = tf.argmax(logits, axis=-1)
        
    # If temp is not 0 then next_token is sampled out of logits
    else:
        logits = logits / temperature
        next_token = tf.random.categorical(logits, num_samples=1)
    
    # Trim dimensions of size 1
    logits = tf.squeeze(logits) 
    next_token = tf.squeeze(next_token) 
    
    # Get the logit of the selected next_token
    logit = logits[next_token].numpy()
    
    # Reshape to (1,1) since this is the expected shape for text encoded as TF tensors
    next_token = tf.reshape(next_token, shape=(1,1)) # shape=(1, 1)
    
    # Dừng lại khi next_token là EOS
    # If next_token is End-of-Sentence token you are done
    if next_token == eos_id:
        done = True
    
    return next_token, logit, state, done


def translate(model, text, max_length=50, temperature=0.0): # Xét câu text="I love languages"
    # Lists to save tokens and logits
    tokens, logits = [], []
    
    # PROCESS THE SENTENCE TO TRANSLATE
    
    # Convert the original string into a tensor
    text = tf.convert_to_tensor(text)[tf.newaxis]
    
    # Vectorize the text using the correct vectorizer
    context = english_vectorizer(text).to_tensor() # [[2 6 150 555 3]], shape=(1, 5), 2 là id của SOS và 3 là id của EOS
    
    # Đưa context qua Encoder
    # Get the encoded context (pass the context through the encoder)
    # Hint: Remember you can get the encoder by using model.encoder
    context = model.encoder(context) # shape=(5, 256)
    
    # INITIAL STATE OF THE DECODER
    
    # First token should be SOS token with shape (1,1)
    next_token = tf.fill((1, 1), sos_id) # [[2]], là id của SOS
    
    # Initial hidden and cell states should be tensors of zeros with shape (1, UNITS)
    state = [tf.zeros((1, UNITS)), tf.zeros((1, UNITS))] # hai states có shape (1, 256)
    
    
    # You are done when you draw a EOS token as next token (initial state is False)
    done = False

    # Iterate for max_length iterations
    for i in range(max_length):
        # Generate the next token
        try:
            # Xem tiếp ở generate_next_token
            next_token, logit, state, done = generate_next_token(
                decoder=model.decoder,
                context=context,
                next_token=next_token,
                done=done,
                state=state,
                temperature=temperature
            )
        except: 
             raise Exception("Problem generating the next token")
        
        # If done then break out of the loop
        if done:
            break
        
        # Add next_token to the list of tokens
        tokens.append(next_token)
        
        # Add logit to the list of logits
        logits.append(logit)
    
    # Gộp tất cả next_token lại rồi chuyển thành text
    # Concatenate all tokens into a tensor
    tokens = tf.concat(tokens, axis=-1)
    
    # Convert the translated tokens into text
    translation = tf.squeeze(tokens_to_text(tokens, id_to_word))
    translation = translation.numpy().decode()
    
    return translation, logits[-1], tokens

In [309]:
# Running this cell multiple times should return the same output since temp is 0
temp = 0.0 
original_sentence = "I love languages"

translation, logit, tokens = translate(model, original_sentence, temperature=temp)

print(f"Temperature: {temp}\n\nOriginal sentence: {original_sentence}\nTranslation: {translation}\nTranslation tokens:{tokens}\nLogit: {logit:.3f}")

Temperature: 0.0

Original sentence: I love languages
Translation: eu [UNK] .
Translation tokens:[[9 1 4]]
Logit: -1.032
